<a href="https://colab.research.google.com/github/sayedHoosini/AI_Portfolio_ITAI2372/blob/main/WiDS_Global_Datathon_2026_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Step 1: Import necessary libraries

import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier

print("Step 1 complete: Libraries imported successfully.")

Step 1 complete: Libraries imported successfully.


In [ ]:
# Step 2: Upload Kaggle dataset files

from google.colab import files

uploaded = files.upload()

Saving metaData.csv to metaData.csv
Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv


In [ ]:
# Step 3: Load the dataset

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")
metadata = pd.read_csv("metaData.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)
print("Metadata shape:", metadata.shape)

Train shape: (221, 37)
Test shape: (95, 35)
Sample submission shape: (95, 5)
Metadata shape: (37, 6)


In [ ]:
# Step 4: Look at the training data

train.head()

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,11757157,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,11945086,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,12044083,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,12052347,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0


I created four target variables based on the wildfire event indicator and time-to-hit information. These targets represent whether a wildfire reaches an evacuation zone within 12, 24, 48, and 72 hours. This step transforms the survival analysis problem into multiple classification tasks.

In [ ]:
# Step 5: Create target variables

train["target_12h"] = ((train["event"] == 1) & (train["time_to_hit_hours"] <= 12)).astype(int)
train["target_24h"] = ((train["event"] == 1) & (train["time_to_hit_hours"] <= 24)).astype(int)
train["target_48h"] = ((train["event"] == 1) & (train["time_to_hit_hours"] <= 48)).astype(int)
train["target_72h"] = ((train["event"] == 1) & (train["time_to_hit_hours"] <= 72)).astype(int)

# Show results
train[["event", "time_to_hit_hours", "target_12h", "target_24h", "target_48h", "target_72h"]].head(10)

,event,time_to_hit_hours,target_12h,target_24h,target_48h,target_72h
0,0,18.892512,0,0,0,0
1,1,22.048108,0,1,1,1
2,1,0.888895,1,1,1,1
3,0,60.953021,0,0,0,0
4,0,44.990274,0,0,0,0
5,0,44.026384,0,0,0,0
6,1,14.845392,0,1,1,1
7,0,66.767927,0,0,0,0
8,0,39.912630,0,0,0,0
9,1,54.638109,0,0,0,1


I prepared the dataset by removing identifiers and target columns, keeping only relevant wildfire features. I also handled missing values using median imputation to ensure the model can train properly.

In [ ]:
# Step 6: Prepare features

# Columns to remove
drop_cols = [
    "event_id",
    "time_to_hit_hours",
    "event",
    "target_12h",
    "target_24h",
    "target_48h",
    "target_72h"
]

# Select features
feature_cols = [col for col in train.columns if col not in drop_cols]

X = train[feature_cols]
X_test = test[feature_cols]

# Fill missing values
X = X.fillna(X.median(numeric_only=True))
X_test = X_test.fillna(X.median(numeric_only=True))

# Show result
print("Number of features:", len(feature_cols))
print("Training shape:", X.shape)
print("Test shape:", X_test.shape)

Number of features: 34
Training shape: (221, 34)
Test shape: (95, 34)


I trained Random Forest models to predict wildfire threat probabilities at 12, 24, 48, and 72 hours. Random Forest was selected because it handles tabular data well and captures non-linear relationships between wildfire features such as growth, distance, and movement.

In [ ]:
# Step 7: Train models

targets = {
    "prob_12h": "target_12h",
    "prob_24h": "target_24h",
    "prob_48h": "target_48h",
    "prob_72h": "target_72h"
}

predictions = {}

for prob_col, target_col in targets.items():
    print("Training model for:", prob_col)

    y = train[target_col]

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        min_samples_leaf=5,
        random_state=42,
        class_weight="balanced"
    )

    model.fit(X, y)

    predictions[prob_col] = model.predict_proba(X_test)[:, 1]

print("All models trained successfully.")

Training model for: prob_12h
Training model for: prob_24h
Training model for: prob_48h
Training model for: prob_72h
All models trained successfully.


In [ ]:
# Step 8: Create submission file

submission = sample_submission.copy()

submission["prob_12h"] = predictions["prob_12h"]
submission["prob_24h"] = predictions["prob_24h"]
submission["prob_48h"] = predictions["prob_48h"]
submission["prob_72h"] = predictions["prob_72h"]

submission.head()

,event_id,prob_12h,prob_24h,prob_48h,prob_72h
0,10662602,0.176371,0.195173,0.183566,0.209048
1,13353600,0.523744,0.725033,0.719979,0.706424
2,13942327,0.101431,0.132628,0.158699,0.175148
3,16112781,0.617609,0.733186,0.746169,0.740512
4,17132808,0.682355,0.659567,0.649696,0.625212


In [ ]:
# Step 9: Download submission file

submission.to_csv("submission.csv", index=False)

from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>